In [7]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/bhavnoorsingh123/cs776-dataset-2-0/vid_2_sub33.avi
/kaggle/input/datasets/bhavnoorsingh123/cs776-dataset-2-0/ground_truth_2_sub46.txt
/kaggle/input/datasets/bhavnoorsingh123/cs776-dataset-2-0/ground_truth_2_sub33.txt
/kaggle/input/datasets/bhavnoorsingh123/cs776-dataset-2-0/vid_2_sub46.avi
/kaggle/input/datasets/bhavnoorsingh123/cs776-dataset-2-0/vid_2_sub10.avi
/kaggle/input/datasets/bhavnoorsingh123/cs776-dataset-2-0/ground_truth_1_11gt.txt
/kaggle/input/datasets/bhavnoorsingh123/cs776-dataset-2-0/vid_2_sub18.avi
/kaggle/input/datasets/bhavnoorsingh123/cs776-dataset-2-0/vid_2_sub32.avi
/kaggle/input/datasets/bhavnoorsingh123/cs776-dataset-2-0/ground_truth_2_sub38.txt
/kaggle/input/datasets/bhavnoorsingh123/cs776-dataset-2-0/ground_truth_2_sub4.txt
/kaggle/input/datasets/bhavnoorsingh123/cs776-dataset-2-0/vid_2_sub24.avi
/kaggle/input/datasets/bhavnoorsingh123/cs776-dataset-2-0/ground_truth_2_sub22.txt
/kaggle/input/datasets/bhavnoorsingh123/cs776-dataset-2-0/vi

In [8]:
!pip install -q onnxruntime-gpu ultralytics

In [9]:
import os
import cv2
import numpy as np
from scipy.interpolate import interp1d
import glob
from ultralytics import YOLO

INPUT_DIR = "/kaggle/input/datasets/bhavnoorsingh123/cs776-dataset-2-0"
OUTPUT_DIR = "/kaggle/working/preprocessed_data"
YOLO_MODEL_PATH = "/kaggle/input/datasets/jayantm23/model-yolo/model.onnx" 

FACE_CROP_SIZE = 72
WINDOW_SIZE = 128

os.makedirs(OUTPUT_DIR, exist_ok=True)
face_model = YOLO(YOLO_MODEL_PATH, task='detect')

def process_and_save_data():
    videos = glob.glob(os.path.join(INPUT_DIR, "*.avi"))
    videos = [v for v in videos if "sub21" not in v and "sub27" not in v]
    
    if len(videos) == 0:
        print(f"No videos found in {INPUT_DIR}.")
        return

    window_count = 0
    
    for vid_path in videos:
        vid_file = os.path.basename(vid_path)
        print(f"Processing: {vid_file}")
        
        if "vid_1_" in vid_file:
            gt_file = vid_file.replace("vid_1_", "gtdump_1_").replace(".avi", ".txt")
        else:
            gt_file = vid_file.replace("vid_2_", "ground_truth_2_").replace(".avi", ".txt")
            
        gt_path = os.path.join(INPUT_DIR, gt_file)
        
        if not os.path.exists(gt_path):
            continue
            
        cap = cv2.VideoCapture(vid_path)
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        frames_list = []
        last_box = None
        alpha = 0.8
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            
            results = face_model(frame, verbose=False)
            
            if len(results[0].boxes) > 0:
                box = results[0].boxes[0].xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = map(int, box)
                x, y, w, h = x1, y1, x2 - x1, y2 - y1
                
                if last_box is None: 
                    last_box = [x, y, w, h]
                else: 
                    last_box = [int(alpha * l + (1 - alpha) * c) for l, c in zip(last_box, [x, y, w, h])]
                
                x, y, w, h = last_box
                x, y = max(0, x), max(0, y) 
                
                try:
                    face_crop = cv2.resize(frame[y:y+h, x:x+w], (FACE_CROP_SIZE, FACE_CROP_SIZE))
                    frames_list.append(cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB))
                except cv2.error:
                    frames_list.append(frames_list[-1] if frames_list else np.zeros((FACE_CROP_SIZE, FACE_CROP_SIZE, 3), dtype=np.uint8))
            else:
                frames_list.append(frames_list[-1] if frames_list else np.zeros((FACE_CROP_SIZE, FACE_CROP_SIZE, 3), dtype=np.uint8))
                
        cap.release()
        
        frames_array = np.array(frames_list)
        if len(frames_array) < WINDOW_SIZE: 
            continue
        
        if "gtdump" in gt_file:
            try:
                data = np.loadtxt(gt_path, delimiter=',')
                gt_ppg = data[:, 3]
                gt_time = data[:, 0] / 1000.0 
            except Exception:
                continue
        else:
            with open(gt_path, 'r') as f:
                lines = f.readlines()
            gt_ppg = np.array([float(x) for x in lines[0].strip().split()])
            gt_time = np.array([float(x) for x in lines[2].strip().split()])
            
        video_timestamps = np.linspace(0, total_frames / fps, len(frames_array))
        gt_time = gt_time - gt_time[0]
        
        _, unique_indices = np.unique(gt_time, return_index=True)
        gt_time_clean = gt_time[unique_indices]
        gt_ppg_clean = gt_ppg[unique_indices]
        
        try:
            interp_func = interp1d(gt_time_clean, gt_ppg_clean, kind='cubic', fill_value="extrapolate")
            aligned_ppg = interp_func(video_timestamps)
        except Exception:
            continue
            
        for start_idx in range(0, len(frames_array) - WINDOW_SIZE, WINDOW_SIZE // 2):
            win_frames = frames_array[start_idx:start_idx+WINDOW_SIZE]
            win_ppg = aligned_ppg[start_idx:start_idx+WINDOW_SIZE]
            
            np.save(os.path.join(OUTPUT_DIR, f"frames_{window_count}.npy"), win_frames)
            np.save(os.path.join(OUTPUT_DIR, f"ppg_{window_count}.npy"), win_ppg)
            window_count += 1
            
process_and_save_data()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Processing: vid_2_sub33.avi
Loading /kaggle/input/datasets/jayantm23/model-yolo/model.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.24.4 with CUDAExecutionProvider
Processing: vid_2_sub46.avi
Processing: vid_2_sub10.avi
Processing: vid_2_sub18.avi
Processing: vid_2_sub32.avi
Processing: vid_2_sub24.avi
Processing: vid_2_sub12.avi
Processing: vid_2_sub40.avi
Processing: vid_2_sub43.avi
Processing: vid_2_sub36.avi
Processing: vid_2_sub34.avi
Processing: vid_2_sub22.avi
Processing: vid_1_11gt.avi
Processing: vid_1_6gt.avi
Processing: vid_2_sub25.avi
Processing: vid_2_sub3.avi
Processing: vid_2_sub5.avi
Processing: vid_1_after_exercise.avi
Processing: vid_2_sub14.avi
Process

In [12]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from scipy.signal import periodogram
from scipy.stats import pearsonr

OUTPUT_DIR = "/kaggle/working/preprocessed_data"
BATCH_SIZE = 8
EPOCHS = 40
LEARNING_RATE = 1e-4

class DiskUBFCDataset(Dataset):
    def __init__(self, data_dir):
        self.frame_files = sorted(glob.glob(os.path.join(data_dir, "frames_*.npy")))
        self.ppg_files = sorted(glob.glob(os.path.join(data_dir, "ppg_*.npy")))
        
    def __len__(self):
        return len(self.frame_files)
        
    def __getitem__(self, idx):
        frames = np.load(self.frame_files[idx]).astype(np.float32) / 255.0
        frames = np.transpose(frames, (3, 0, 1, 2))
        ppg = np.load(self.ppg_files[idx]).astype(np.float32)
        ppg = (ppg - np.mean(ppg)) / (np.std(ppg) + 1e-8)
        return torch.tensor(frames), torch.tensor(ppg)

def calculate_bpm_from_signal(signal, fps=30.0):
    freqs, psd = periodogram(signal, fs=fps, nfft=1024, detrend='constant')
    valid_indices = np.where((freqs >= 0.75) & (freqs <= 2.5))[0]
    
    if len(valid_indices) == 0:
        return 75.0 
        
    max_idx = np.argmax(psd[valid_indices])
    hr_hz = freqs[valid_indices][max_idx]
    return hr_hz * 60.0 

def evaluate_batch_metrics(predictions, targets, fps=30.0):
    preds_np = predictions.detach().cpu().numpy()
    targets_np = targets.detach().cpu().numpy()
    
    pred_bpms, target_bpms = [], []
    for i in range(preds_np.shape[0]):
        pred_bpms.append(calculate_bpm_from_signal(preds_np[i], fps))
        target_bpms.append(calculate_bpm_from_signal(targets_np[i], fps))
        
    p_arr, t_arr = np.array(pred_bpms), np.array(target_bpms)
    mae = np.mean(np.abs(p_arr - t_arr))
    rmse = np.sqrt(np.mean((p_arr - t_arr)**2))
    
    return mae, rmse, pred_bpms, target_bpms

class SEBlock3D(nn.Module):
    def __init__(self, channel, reduction=16):
        super(SEBlock3D, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1, 1)
        return x * y.expand_as(x)

class ResBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResBlock3D, self).__init__()
        self.conv1 = nn.Conv3d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm3d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv3d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm3d(out_channels)
        self.se = SEBlock3D(out_channels)
        
        self.downsample = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv3d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm3d(out_channels)
            )

    def forward(self, x):
        identity = self.downsample(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.se(out)
        out += identity
        out = self.relu(out)
        return out

class AdvancedPhysNet(nn.Module):
    def __init__(self):
        super(AdvancedPhysNet, self).__init__()
        self.stem = nn.Sequential(
            nn.Conv3d(3, 32, kernel_size=(1, 5, 5), stride=1, padding=(0, 2, 2)),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool3d((1, 2, 2), stride=(1, 2, 2))
        )
        
        self.layer1 = ResBlock3D(32, 64, stride=(1, 2, 2))
        self.layer2 = ResBlock3D(64, 64, stride=1)
        self.layer3 = ResBlock3D(64, 128, stride=(1, 2, 2))
        
        self.adaptive_pool = nn.AdaptiveAvgPool3d((None, 1, 1))
        self.temporal_conv = nn.Conv1d(128, 64, kernel_size=3, padding=1)
        self.regressor = nn.Conv1d(64, 1, kernel_size=1)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        
        x = self.adaptive_pool(x).squeeze(-1).squeeze(-1)
        
        x = F.relu(self.temporal_conv(x))
        
        # FIX: Squeeze out the channel dimension so output is [Batch, 128] instead of [Batch, 1, 128]
        x = self.regressor(x).squeeze(1) 
        
        return x

class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.5):
        super(CombinedLoss, self).__init__()
        self.alpha = alpha
        self.l1 = nn.SmoothL1Loss()

    def forward(self, preds, targets):
        p_mean = torch.mean(preds, dim=1, keepdim=True)
        t_mean = torch.mean(targets, dim=1, keepdim=True)
        cov = torch.sum((preds - p_mean) * (targets - t_mean), dim=1)
        p_var = torch.sqrt(torch.sum((preds - p_mean)**2, dim=1) + 1e-8)
        t_var = torch.sqrt(torch.sum((targets - t_mean)**2, dim=1) + 1e-8)
        pearson_loss = 1.0 - torch.mean(cov / (p_var * t_var))
        
        l1_loss = self.l1(preds, targets)
        
        return pearson_loss + self.alpha * l1_loss

def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dataset = DiskUBFCDataset(OUTPUT_DIR)
    
    if len(dataset) == 0:
        return

    dataset_size = len(dataset)
    train_size = int(0.7 * dataset_size)
    val_size = int(0.15 * dataset_size)
    test_size = dataset_size - train_size - val_size

    train_ds, val_ds, test_ds = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])
    
    torch.save(test_ds.indices, "/kaggle/working/test_indices.pt")
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = AdvancedPhysNet().to(device)
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
        
    criterion = CombinedLoss().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4)
    scaler = torch.amp.GradScaler('cuda') 

    best_val_mae = float('inf')

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        
        for b_idx, (frames, ppg) in enumerate(train_loader):
            frames, ppg = frames.to(device), ppg.to(device)
            optimizer.zero_grad()
            
            with torch.amp.autocast('cuda'):
                preds = model(frames)
                loss = criterion(preds, ppg)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            running_loss += loss.item()
            if b_idx % 10 == 0:
                print(f"Epoch [{epoch+1}/{EPOCHS}], Step [{b_idx}/{len(train_loader)}], Loss: {loss.item():.4f}")

        model.eval()
        val_loss, val_mae, val_rmse = 0.0, 0.0, 0.0
        all_p_bpms, all_t_bpms = [], []
        
        with torch.no_grad():
            for frames, ppg in val_loader:
                frames, ppg = frames.to(device), ppg.to(device)
                
                with torch.amp.autocast('cuda'):
                    preds = model(frames)
                    loss = criterion(preds, ppg)
                
                val_loss += loss.item()
                b_mae, b_rmse, p_bpm, t_bpm = evaluate_batch_metrics(preds, ppg)
                val_mae += b_mae
                val_rmse += b_rmse
                
                all_p_bpms.extend(p_bpm)
                all_t_bpms.extend(t_bpm)
                
        avg_v_loss = val_loss / len(val_loader)
        avg_v_mae = val_mae / len(val_loader)
        avg_v_rmse = val_rmse / len(val_loader)
        
        if np.std(all_p_bpms) > 0 and np.std(all_t_bpms) > 0:
            pearson_r, _ = pearsonr(all_p_bpms, all_t_bpms)
        else:
            pearson_r = 0.0
            
        scheduler.step(avg_v_loss)
        
        print(f"Val Loss: {avg_v_loss:.4f} | Val MAE: {avg_v_mae:.2f} | Val RMSE: {avg_v_rmse:.2f} | Pearson: {pearson_r:.4f}")
        
        if avg_v_mae < best_val_mae:
            best_val_mae = avg_v_mae
            torch.save(model.state_dict(), "/kaggle/working/best_physnet_weights.pth")

    torch.save(model.state_dict(), "/kaggle/working/final_physnet_weights.pth")

if __name__ == "__main__":
    train()

Epoch [1/40], Step [0/104], Loss: 1.2361
Epoch [1/40], Step [10/104], Loss: 1.1722
Epoch [1/40], Step [20/104], Loss: 1.1763
Epoch [1/40], Step [30/104], Loss: 1.1699
Epoch [1/40], Step [40/104], Loss: 1.1227
Epoch [1/40], Step [50/104], Loss: 1.1979
Epoch [1/40], Step [60/104], Loss: 1.0929
Epoch [1/40], Step [70/104], Loss: 1.1850
Epoch [1/40], Step [80/104], Loss: 1.1333
Epoch [1/40], Step [90/104], Loss: 1.0861
Epoch [1/40], Step [100/104], Loss: 1.0428
Val Loss: 1.1468 | Val MAE: 23.30 | Val RMSE: 30.56 | Pearson: 0.2099
Epoch [2/40], Step [0/104], Loss: 1.1059
Epoch [2/40], Step [10/104], Loss: 0.9888
Epoch [2/40], Step [20/104], Loss: 1.0136
Epoch [2/40], Step [30/104], Loss: 0.9640
Epoch [2/40], Step [40/104], Loss: 0.9355
Epoch [2/40], Step [50/104], Loss: 0.9472
Epoch [2/40], Step [60/104], Loss: 0.9671
Epoch [2/40], Step [70/104], Loss: 1.1088
Epoch [2/40], Step [80/104], Loss: 1.1849
Epoch [2/40], Step [90/104], Loss: 0.9934
Epoch [2/40], Step [100/104], Loss: 0.9628
Val Lo

In [13]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from scipy.signal import periodogram
from scipy.stats import pearsonr

OUTPUT_DIR = "/kaggle/working/preprocessed_data"
BATCH_SIZE = 8

class DiskUBFCDataset(Dataset):
    def __init__(self, data_dir):
        self.frame_files = sorted(glob.glob(os.path.join(data_dir, "frames_*.npy")))
        self.ppg_files = sorted(glob.glob(os.path.join(data_dir, "ppg_*.npy")))
        
    def __len__(self):
        return len(self.frame_files)
        
    def __getitem__(self, idx):
        frames = np.load(self.frame_files[idx]).astype(np.float32) / 255.0
        frames = np.transpose(frames, (3, 0, 1, 2))
        ppg = np.load(self.ppg_files[idx]).astype(np.float32)
        ppg = (ppg - np.mean(ppg)) / (np.std(ppg) + 1e-8)
        return torch.tensor(frames), torch.tensor(ppg)

def calculate_bpm_from_signal(signal, fps=30.0):
    freqs, psd = periodogram(signal, fs=fps, nfft=1024, detrend='constant')
    valid_indices = np.where((freqs >= 0.75) & (freqs <= 2.5))[0]
    
    if len(valid_indices) == 0:
        return 75.0 
        
    max_idx = np.argmax(psd[valid_indices])
    hr_hz = freqs[valid_indices][max_idx]
    return hr_hz * 60.0 

def evaluate_batch_metrics(predictions, targets, fps=30.0):
    preds_np = predictions.detach().cpu().numpy()
    targets_np = targets.detach().cpu().numpy()
    
    pred_bpms, target_bpms = [], []
    for i in range(preds_np.shape[0]):
        pred_bpms.append(calculate_bpm_from_signal(preds_np[i], fps))
        target_bpms.append(calculate_bpm_from_signal(targets_np[i], fps))
        
    p_arr, t_arr = np.array(pred_bpms), np.array(target_bpms)
    mae = np.mean(np.abs(p_arr - t_arr))
    rmse = np.sqrt(np.mean((p_arr - t_arr)**2))
    
    return mae, rmse, pred_bpms, target_bpms

class SEBlock3D(nn.Module):
    def __init__(self, channel, reduction=16):
        super(SEBlock3D, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1, 1)
        return x * y.expand_as(x)

class ResBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResBlock3D, self).__init__()
        self.conv1 = nn.Conv3d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm3d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv3d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm3d(out_channels)
        self.se = SEBlock3D(out_channels)
        
        self.downsample = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv3d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm3d(out_channels)
            )

    def forward(self, x):
        identity = self.downsample(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.se(out)
        out += identity
        out = self.relu(out)
        return out

class AdvancedPhysNet(nn.Module):
    def __init__(self):
        super(AdvancedPhysNet, self).__init__()
        self.stem = nn.Sequential(
            nn.Conv3d(3, 32, kernel_size=(1, 5, 5), stride=1, padding=(0, 2, 2)),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool3d((1, 2, 2), stride=(1, 2, 2))
        )
        
        self.layer1 = ResBlock3D(32, 64, stride=(1, 2, 2))
        self.layer2 = ResBlock3D(64, 64, stride=1)
        self.layer3 = ResBlock3D(64, 128, stride=(1, 2, 2))
        
        self.adaptive_pool = nn.AdaptiveAvgPool3d((None, 1, 1))
        self.temporal_conv = nn.Conv1d(128, 64, kernel_size=3, padding=1)
        self.regressor = nn.Conv1d(64, 1, kernel_size=1)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        
        x = self.adaptive_pool(x).squeeze(-1).squeeze(-1)
        x = F.relu(self.temporal_conv(x))
        x = self.regressor(x).squeeze(1)
        
        return x

def evaluate_test_set():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Evaluating on: {device}")
    
    full_dataset = DiskUBFCDataset(OUTPUT_DIR)
    
    try:
        test_indices = torch.load("/kaggle/working/test_indices.pt")
    except FileNotFoundError:
        print("test_indices.pt not found. Run training first.")
        return
        
    test_ds = Subset(full_dataset, test_indices)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    model = AdvancedPhysNet().to(device)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
        
    try:
        model.load_state_dict(torch.load("/kaggle/working/best_physnet_weights.pth"))
    except FileNotFoundError:
        print("best_physnet_weights.pth not found. Run training first.")
        return
        
    model.eval()
    
    total_mae, total_rmse = 0.0, 0.0
    all_p_bpms, all_t_bpms = [], []
    
    with torch.no_grad():
        for frames, ppg in test_loader:
            frames, ppg = frames.to(device), ppg.to(device)
            
            with torch.amp.autocast('cuda'):
                preds = model(frames)
                
            b_mae, b_rmse, p_bpm, t_bpm = evaluate_batch_metrics(preds, ppg)
            total_mae += b_mae
            total_rmse += b_rmse
            
            all_p_bpms.extend(p_bpm)
            all_t_bpms.extend(t_bpm)
            
    final_mae = total_mae / len(test_loader)
    final_rmse = total_rmse / len(test_loader)
    
    if np.std(all_p_bpms) > 0 and np.std(all_t_bpms) > 0:
        pearson_r, _ = pearsonr(all_p_bpms, all_t_bpms)
    else:
        pearson_r = 0.0
        
    print("\n=== FINAL TEST SET RESULTS ===")
    print(f"Total Test Windows: {len(test_ds)}")
    print(f"Test MAE:  {final_mae:.2f} BPM")
    print(f"Test RMSE: {final_rmse:.2f} BPM")
    print(f"Pearson r: {pearson_r:.4f}")
    print("==============================\n")

if __name__ == "__main__":
    evaluate_test_set()

Evaluating on: cuda

=== FINAL TEST SET RESULTS ===
Total Test Windows: 179
Test MAE:  4.24 BPM
Test RMSE: 7.41 BPM
Pearson r: 0.8407

